# 01 — Exploración y preparación inicial de BDD100K

Este notebook prepara el entorno en Google Colab, descarga los paquetes de imágenes de 10K y las anotaciones de segmentación semántica y por instancias, comprueba la estructura y genera un informe exploratorio. **No descarga los videos de 1.8 TB ni entrena modelos.**

> BDD100K requiere aceptar sus condiciones oficiales de uso. Consulta la [documentación de descarga](https://github.com/bdd100k/bdd100k/blob/master/doc/source/download.rst). Los archivos se guardan en Google Drive para persistir entre sesiones. La descarga puede tardar y depende de la disponibilidad de Colab/red.

## 1. Conectar Google Drive
Ejecuta esta celda y autoriza el acceso a tu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_DIR = Path('/content/drive/MyDrive/Imagenes_Tema2')
DOWNLOAD_DIR = PROJECT_DIR / 'downloads'
DATA_DIR = PROJECT_DIR / 'data' / 'bdd100k'
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
print('Proyecto:', PROJECT_DIR)


## 2. Descargar los paquetes necesarios
Se descargan los tres splits de imágenes 10K y las etiquetas semánticas y de instancias. Las imágenes 10K son el conjunto indicado por BDD100K para estas dos tareas. No descargar `100k_images_*` para este primer flujo: carriles y área transitable usan anotaciones asociadas al conjunto de 100K, no al split 10K.

In [ ]:
# CONFIGURACIÓN DEL DIRECTORIO DE DESCARGA

from pathlib import Path

DOWNLOAD_DIR = Path("/content/bdd100k_download")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio de descarga:", DOWNLOAD_DIR)

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "http://128.32.162.150/bdd100k/"

response = requests.get(BASE_URL, timeout=30)
soup = BeautifulSoup(response.text, "html.parser")

archivos_objetivo = [
    "bdd100k_images_10k.zip",
    "bdd100k_seg_maps.zip",
    "bdd100k_labels.zip",
    "bdd100k_info.zip"
]

for nombre in archivos_objetivo:
    url = urljoin(BASE_URL, nombre)
    r = requests.head(url, timeout=30)

    print(f"\nArchivo: {nombre}")
    print("HTTP:", r.status_code)
    print("Tamaño:", r.headers.get("Content-Length", "No disponible"))

In [ ]:

# BLOQUE 2 — DESCARGA BDD100K DESDE BERKELEY CON PROGRESO

import requests
import time
from pathlib import Path
from IPython.display import display, clear_output

BASE_URL = "http://128.32.162.150/bdd100k/"

FILES = [
    "bdd100k_images_10k.zip",
    "bdd100k_seg_maps.zip",
    "bdd100k_labels.zip"
]

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)


def format_size(size):
    return f"{size / (1024**3):.2f} GB"


def download_file(filename, retries=3):

    url = BASE_URL + filename
    target = DOWNLOAD_DIR / filename

    for attempt in range(1, retries + 1):

        try:
            print(f"\nDescargando: {filename} ({attempt}/{retries})")

            downloaded = target.stat().st_size if target.exists() else 0

            headers = {}

            if downloaded > 0:
                headers["Range"] = f"bytes={downloaded}-"

            with requests.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 180)
            ) as response:

                if downloaded > 0 and response.status_code == 206:
                    mode = "ab"

                elif response.status_code == 200:
                    mode = "wb"
                    downloaded = 0

                else:
                    response.raise_for_status()
                    mode = "wb"

                total = int(response.headers.get(
                    "Content-Length", 0
                )) + downloaded

                start_time = time.time()
                last_update = start_time

                with open(target, mode) as f:

                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):

                        if not chunk:
                            continue

                        f.write(chunk)
                        downloaded += len(chunk)

                        now = time.time()

                        if now - last_update >= 0.5:

                            elapsed = now - start_time
                            speed = (
                                downloaded - (target.stat().st_size
                                - downloaded if False else 0)
                            ) / max(elapsed, 0.1)

                            if total > 0:
                                percentage = downloaded / total * 100
                                remaining = total - downloaded
                                eta = remaining / max(speed, 1)

                                bar_length = 30
                                filled = int(
                                    bar_length * downloaded / total
                                )

                                bar = (
                                    "█" * filled +
                                    "░" * (bar_length - filled)
                                )

                                print(
                                    f"\r[{bar}] {percentage:6.2f}% | "
                                    f"{format_size(downloaded)} / "
                                    f"{format_size(total)} | "
                                    f"ETA: {int(eta)} s",
                                    end=""
                                )

                            else:
                                print(
                                    f"\rDescargado: "
                                    f"{format_size(downloaded)}",
                                    end=""
                                )

                            last_update = now

                actual_size = target.stat().st_size

                if total and actual_size != total:
                    raise RuntimeError(
                        f"Tamaño incompleto: {actual_size} de {total}"
                    )

                print(f"\n\n¡Descarga completada: {filename}!")
                print(f"Tamaño final: {format_size(actual_size)}")

                return

        except Exception as e:

            print("\nError:", e)
            time.sleep(5 * attempt)

    raise RuntimeError(f"No se pudo descargar {filename}")


for filename in FILES:
    download_file(filename)

print("\n¡DESCARGA DE BDD100K COMPLETADA!")


## 3. Verificar checksums de las etiquetas
Los MD5 siguientes corresponden a los paquetes de etiquetas indicados por la documentación oficial. No se fuerza una suma para los ZIP de imágenes porque la documentación consultada no publica aquí un MD5 individual por split.

In [ ]:
# DIAGNÓSTICO DEL ZIP DE IMÁGENES

import zipfile
import hashlib
from pathlib import Path

archivo = DOWNLOAD_DIR / "bdd100k_images_10k.zip"

print("¿Existe el archivo?:", archivo.exists())

if archivo.exists():

    print("Tamaño:", archivo.stat().st_size, "bytes")
    print("Tamaño en GB:", round(archivo.stat().st_size / 1024**3, 3))

    with open(archivo, "rb") as f:
        print("Primeros bytes:", f.read(8))

    print("\n¿Es un ZIP válido?:", zipfile.is_zipfile(archivo))

    if zipfile.is_zipfile(archivo):
        with zipfile.ZipFile(archivo, "r") as z:
            print("Número de elementos:", len(z.namelist()))
            print("Primeros archivos:", z.namelist()[:10])
            print("Comprobando integridad interna...")
            print("Resultado:", z.testzip())

    md5 = hashlib.md5()

    with open(archivo, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            md5.update(chunk)

    print("\nMD5:", md5.hexdigest())

In [ ]:
# BLOQUE 3 — VERIFICACIÓN DE INTEGRIDAD DE LOS ZIP

import hashlib
import zipfile

FILES = [
    "bdd100k_images_10k.zip",
    "bdd100k_seg_maps.zip",
    "bdd100k_labels.zip"
]

for filename in FILES:

    archivo = DOWNLOAD_DIR / filename

    print("\nComprobando:", filename)

    if not archivo.exists():
        print("NO EXISTE")
        continue

    md5 = hashlib.md5()

    with open(archivo, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            md5.update(chunk)

    checksum = md5.hexdigest()

    valido = zipfile.is_zipfile(archivo)

    print("Tamaño:", archivo.stat().st_size)
    print("MD5:", checksum)
    print("ZIP válido:", valido)

    if valido:
        with zipfile.ZipFile(archivo, "r") as z:
            print("Elementos:", len(z.namelist()))
            print("Integridad:", z.testzip())

In [ ]:

# BLOQUE 3 — VERIFICACIÓN DE INTEGRIDAD BDD100K

import hashlib
import zipfile

FILES = {
    "bdd100k_images_10k.zip": "9c5ee9df0097cc1ab799f58feb776611",
    "bdd100k_seg_maps.zip": "7607c97f0fc366fad39f0a450ab7c4fc",
    "bdd100k_labels.zip": "2d0165fd89b11025206e21a25ae1089d"
}

def calculate_md5(path):
    md5 = hashlib.md5()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            md5.update(chunk)

    return md5.hexdigest()


for filename, expected_md5 in FILES.items():

    archivo = DOWNLOAD_DIR / filename

    if not archivo.exists():
        raise FileNotFoundError(f"No existe: {filename}")

    actual_md5 = calculate_md5(archivo)

    print(f"\n{filename}")
    print("MD5:", actual_md5)

    if actual_md5 != expected_md5:
        raise ValueError(f"Checksum incorrecto: {filename}")

    if not zipfile.is_zipfile(archivo):
        raise ValueError(f"ZIP inválido: {filename}")

    with zipfile.ZipFile(archivo, "r") as z:
        error = z.testzip()

        if error is not None:
            raise ValueError(
                f"Archivo dañado dentro de {filename}: {error}"
            )

    print("VERIFICACIÓN CORRECTA")

print("\n¡LOS TRES ARCHIVOS BDD100K ESTÁN VERIFICADOS!")



## 4. Extraer los ZIP
La extracción se hace en Drive para conservar los datos al cerrar la sesión. Puede requerir espacio libre considerable.

In [ ]:
# CONFIGURACIÓN DEL DIRECTORIO DE DATOS

from pathlib import Path

DATA_DIR = Path("/content/bdd100k_dataset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Directorio de extracción:", DATA_DIR)

In [ ]:
def safe_extract(zip_path, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        root = destination.resolve()
        for member in zf.infolist():
            resolved = (destination / member.filename).resolve()
            if root not in resolved.parents and resolved != root:
                raise RuntimeError(f'Ruta insegura en ZIP: {member.filename}')
        zf.extractall(destination)
    print('Extraído:', Path(zip_path).name)
for filename in FILES:
    safe_extract(DOWNLOAD_DIR / filename, DATA_DIR)
print('Extracción terminada en', DATA_DIR)


## 5. Inspeccionar carpetas, conteos y archivos de anotación

In [ ]:
from pathlib import Path

print("DIRECTORIOS PRINCIPALES:")
for p in DATA_DIR.iterdir():
    print(p.name, "->", "CARPETA" if p.is_dir() else "ARCHIVO")

print("\nCARPETAS DE IMÁGENES:")
for p in DATA_DIR.rglob("10k"):
    if p.is_dir():
        print(p)

print("\nARCHIVOS DE IMÁGENES:")
for p in DATA_DIR.rglob("*.jpg"):
    print(p.relative_to(DATA_DIR))
    break

print("\nARCHIVOS JSON:")
json_files = list(DATA_DIR.rglob("*.json"))

print("Total de archivos JSON:", len(json_files))

for p in json_files[:20]:
    print(p.relative_to(DATA_DIR),
          f"({p.stat().st_size / 1024**2:.2f} MB)")

print("\nCOMPROBACIÓN DE DIVISIONES:")
for split in ["train", "val", "test"]:
    matches = list(DATA_DIR.rglob(split))
    for folder in matches:
        if folder.is_dir():
            images = list(folder.glob("*.jpg"))
            if images:
                print(f"{folder}: {len(images)} imágenes")

In [ ]:
from pathlib import Path

for folder_name in ["color_labels", "labels"]:
    root = DATA_DIR / folder_name

    print(f"\n{'='*50}")
    print(f"CONTENIDO DE: {folder_name}")
    print(f"{'='*50}")

    if not root.exists():
        print("Carpeta no encontrada")
        continue

    for p in sorted(root.rglob("*")):
        if p.is_dir():
            print(f"\nCarpeta: {p.relative_to(DATA_DIR)}")
            files = [x for x in p.iterdir() if x.is_file()]
            print("Archivos directos:", len(files))
            print("Ejemplos:", [x.name for x in files[:5]])

In [ ]:
from pathlib import Path

# Rutas reales del dataset
IMAGE_TRAIN_DIR = DATA_DIR / "10k" / "train"
IMAGE_VAL_DIR = DATA_DIR / "10k" / "val"

MASK_TRAIN_DIR = DATA_DIR / "labels" / "train"
MASK_VAL_DIR = DATA_DIR / "labels" / "val"

COLOR_TRAIN_DIR = DATA_DIR / "color_labels" / "train"
COLOR_VAL_DIR = DATA_DIR / "color_labels" / "val"

# Función para obtener el identificador de cada archivo
def get_image_id(filename):
    return filename.replace("_train_id.png", "").replace("_train_color.png", "")

# Comprobar correspondencia entre imágenes y máscaras
for split, image_dir, mask_dir, color_dir in [
    ("train", IMAGE_TRAIN_DIR, MASK_TRAIN_DIR, COLOR_TRAIN_DIR),
    ("val", IMAGE_VAL_DIR, MASK_VAL_DIR, COLOR_VAL_DIR)
]:
    images = {p.stem for p in image_dir.glob("*.jpg")}
    masks = {get_image_id(p.name) for p in mask_dir.glob("*.png")}
    colors = {get_image_id(p.name) for p in color_dir.glob("*.png")}

    print(f"\n--- {split.upper()} ---")
    print("Imágenes:", len(images))
    print("Máscaras ID:", len(masks))
    print("Máscaras color:", len(colors))
    print("Imágenes sin máscara ID:", len(images - masks))
    print("Imágenes sin máscara color:", len(images - colors))

## 6. Visualizar una imagen y su máscara semántica (si están presentes)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# Seleccionar una imagen aleatoria del conjunto de entrenamiento
image_files = list(IMAGE_TRAIN_DIR.glob("*.jpg"))
image_path = random.choice(image_files)

# Obtener identificador
image_id = image_path.stem

# Construir rutas de las máscaras
mask_path = MASK_TRAIN_DIR / f"{image_id}_train_id.png"
color_path = COLOR_TRAIN_DIR / f"{image_id}_train_color.png"

# Cargar imágenes
original = Image.open(image_path).convert("RGB")
mask = Image.open(mask_path)
color_mask = Image.open(color_path).convert("RGB")

# Mostrar resultados
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(original)
axes[0].set_title("Imagen original")

axes[1].imshow(mask)
axes[1].set_title("Máscara de identificadores")

axes[2].imshow(color_mask)
axes[2].set_title("Máscara de colores")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Imagen:", image_path.name)
print("Dimensiones originales:", original.size)
print("Dimensiones máscara ID:", mask.size)
print("Dimensiones máscara color:", color_mask.size)

## 7. Metadatos ambientales: alcance y limitación
BDD100K documenta atributos `weather`, `scene` y `timeofday` en los JSON de anotación de detección asociados a las imágenes/vídeos de 100K. Sin embargo, las imágenes 10K usadas para segmentación semántica e instancias **no constituyen un subconjunto exacto** de las 100K. Por tanto, no se debe asignar una condición ambiental por suposición. En una siguiente etapa habrá que obtener los JSON de metadatos, cruzar por nombre/ID de imagen y reportar la cobertura real; las imágenes sin correspondencia quedarán como `unknown`.

## 8. Guardar un informe de exploración

In [ ]:
from pathlib import Path
from datetime import datetime

# Rutas reales del conjunto de datos
image_dirs = {
    "train": DATA_DIR / "10k" / "train",
    "val": DATA_DIR / "10k" / "val",
    "test": DATA_DIR / "10k" / "test"
}

mask_dirs = {
    "train": DATA_DIR / "labels" / "train",
    "val": DATA_DIR / "labels" / "val"
}

color_dirs = {
    "train": DATA_DIR / "color_labels" / "train",
    "val": DATA_DIR / "color_labels" / "val"
}

# Crear informe
report = [
    "# Informe inicial BDD100K",
    f"Fecha de ejecución: {datetime.now().isoformat(timespec='seconds')}",
    f"Ruta de datos: {DATA_DIR}",
    "",
    "## Conteo de imágenes"
]

for split, folder in image_dirs.items():
    count = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
    report.append(f"- {split}: {count} imágenes")

report += [
    "",
    "## Máscaras de segmentación",
]

for split in ["train", "val"]:
    mask_count = len(list(mask_dirs[split].glob("*.png")))
    color_count = len(list(color_dirs[split].glob("*.png")))

    report.append(f"- {split}:")
    report.append(f"  - Máscaras ID: {mask_count}")
    report.append(f"  - Máscaras color: {color_count}")

report += [
    "",
    "## Próximos pasos",
    "- Crear el mapeo de clases de segmentación.",
    "- Preparar los dataloaders de entrenamiento y validación.",
    "- Cruzar metadatos ambientales con los IDs que tengan correspondencia.",
    "- Entrenar primero con un subconjunto pequeño en Colab.",
    "",
    "## Observaciones",
    "- Las imágenes de entrenamiento y validación tienen máscaras correspondientes.",
    "- Las dimensiones de la imagen original y las máscaras fueron verificadas.",
    "- Las imágenes sin correspondencia de máscaras fueron 0 en train y val."
]

# Guardar informe dentro de la carpeta del dataset
report_path = DATA_DIR / "informe_exploracion_bdd100k.md"

report_path.write_text(
    "\n".join(report),
    encoding="utf-8"
)

print("\n".join(report))

print("\nInforme guardado en:", report_path)

## Notas para el siguiente notebook
- Para BDD100K, semantic segmentation e instance segmentation usan los datos de 10K y sus respectivos paquetes de etiquetas.
- Lane marking y drivable area se anotan sobre 100K Images; no deben mezclarse como si fueran máscaras de los 10K.
- El entrenamiento completo de dos arquitecturas puede exceder el tiempo/GPU disponible en Colab gratuito. Empezar con un subconjunto, resolución moderada, batch pequeño y checkpoints en Drive.
- No subir los datos ni los ZIP al repositorio GitHub.